<a href="https://colab.research.google.com/github/nohaelkachach/casiav2-splicing-gradcam-audit/blob/main/06_tp_sensitivity_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 — True-positive-only sensitivity analysis

Restricts the core analysis to spliced images each classifier correctly
predicted as "Tampered" (true positives), separately per architecture, and
checks whether the polarity×size effect (and its architecture-dependent
reversal) survives when limited to confidently-recognized splices.

This is a robustness check, not a required analysis — the main results
already have a documented justification for including all 1,828 images
regardless of prediction correctness (Grad-CAM targets the Tampered-class
score directly, independent of argmax). This notebook asks whether the
same conclusions hold on the narrower, higher-confidence subset.

## Setup — load checkpoints, split, and regression-ready data

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
import statsmodels.formula.api as smf
from scipy.stats import mannwhitneyu
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"

with open("/content/drive/MyDrive/CASIA2.0/casia_split.json") as f:
    split_dict = json.load(f)
spliced_files_all = split_dict["held_out_spliced"]

base = "/content/drive/MyDrive/CASIA2.0/CASIA2.0_revised"
tp_dir = os.path.join(base, "Tp")

val_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

resnet = models.resnet18(weights=None)
resnet.fc = nn.Linear(resnet.fc.in_features, 2)
resnet.load_state_dict(torch.load("/content/drive/MyDrive/CASIA2.0/casia_resnet_best.pt", map_location=device))
resnet = resnet.to(device).eval()

effnet = models.efficientnet_b0(weights=None)
num_features = effnet.classifier[1].in_features
effnet.classifier[1] = nn.Linear(num_features, 2)
effnet.load_state_dict(torch.load("/content/drive/MyDrive/CASIA2.0/casia_effnet_best.pt", map_location=device))
effnet = effnet.to(device).eval()

resnet_df = pd.read_csv("/content/drive/MyDrive/CASIA2.0/regression_ready_resnet.csv")
effnet_df = pd.read_csv("/content/drive/MyDrive/CASIA2.0/regression_ready_effnet.csv")
resnet_df["polarity_bin"] = (resnet_df["polarity"] == "dark_on_bright").astype(int)
effnet_df["polarity_bin"] = (effnet_df["polarity"] == "dark_on_bright").astype(int)

print(f"resnet_df: {len(resnet_df)} rows | effnet_df: {len(effnet_df)} rows")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
resnet_df: 1828 rows | effnet_df: 1828 rows


## Get each model's actual prediction (Tampered vs. Authentic) per image
This is the piece missing from the regression-ready CSVs — `iou`/`auc_iou`
don't tell you whether the classifier's argmax agreed. Computed once per
architecture and merged in as `predicted_label` (1=Tampered, 0=Authentic).

In [4]:
def get_predictions(model, filenames, image_dir):
    model.eval()
    preds = []
    with torch.no_grad():
        for i, fname in enumerate(filenames):
            img = Image.open(os.path.join(image_dir, fname)).convert("RGB")
            input_tensor = val_transform(img).unsqueeze(0).to(device)
            pred = model(input_tensor).argmax(1).item()
            preds.append(pred)
            if (i + 1) % 400 == 0:
                print(f"  {i+1}/{len(filenames)}")
    return preds

print("ResNet18 predictions...")
resnet_preds = get_predictions(resnet, spliced_files_all, tp_dir)
pred_df_resnet = pd.DataFrame({"filename": spliced_files_all, "predicted_label": resnet_preds})

print("\nEfficientNet-B0 predictions...")
effnet_preds = get_predictions(effnet, spliced_files_all, tp_dir)
pred_df_effnet = pd.DataFrame({"filename": spliced_files_all, "predicted_label": effnet_preds})

resnet_df = resnet_df.merge(pred_df_resnet, on="filename", how="left")
effnet_df = effnet_df.merge(pred_df_effnet, on="filename", how="left")

print(f"\nResNet18 TP count: {(resnet_df['predicted_label']==1).sum()} of {len(resnet_df)}")
print(f"EfficientNet-B0 TP count: {(effnet_df['predicted_label']==1).sum()} of {len(effnet_df)}")

ResNet18 predictions...
  400/1828
  800/1828
  1200/1828
  1600/1828

EfficientNet-B0 predictions...
  400/1828
  800/1828
  1200/1828
  1600/1828

ResNet18 TP count: 1576 of 1828
EfficientNet-B0 TP count: 1499 of 1828


## Build TP-only subsets, per architecture
Each architecture's TP subset is independent — the two models don't necessarily
agree on which images they get right, so n will differ between them.

In [5]:
resnet_tp = resnet_df[resnet_df["predicted_label"] == 1].copy()
effnet_tp = effnet_df[effnet_df["predicted_label"] == 1].copy()

print(f"ResNet18: full n={len(resnet_df)} -> TP-only n={len(resnet_tp)} "
      f"({len(resnet_tp)/len(resnet_df)*100:.1f}%)")
print(f"EfficientNet-B0: full n={len(effnet_df)} -> TP-only n={len(effnet_tp)} "
      f"({len(effnet_tp)/len(effnet_df)*100:.1f}%)")

ResNet18: full n=1828 -> TP-only n=1576 (86.2%)
EfficientNet-B0: full n=1828 -> TP-only n=1499 (82.0%)


## Rerun core tests on TP-only subsets
Median-split regression, stratified Mann-Whitney, and raw means — same tests
as the main analysis, now restricted to confidently-recognized splices.

In [6]:
def stratified_mannwhitney(df, outcome="iou"):
    for pct in [0.40, 0.33, 0.30, 0.25, 0.20]:
        cutoff = df["splice_size_frac"].quantile(1 - pct)
        large = df[df["splice_size_frac"] >= cutoff]
        dark = large[large["polarity_bin"] == 1][outcome]
        bright = large[large["polarity_bin"] == 0][outcome]
        if len(dark) < 5 or len(bright) < 5:
            print(f"Top {int(pct*100)}%: skipped (n_dark={len(dark)}, n_bright={len(bright)}, too small)")
            continue
        stat, p = mannwhitneyu(dark, bright, alternative="two-sided")
        r = 1 - (2 * stat) / (len(dark) * len(bright))
        higher = "dark_on_bright" if dark.mean() > bright.mean() else "bright_on_dark"
        print(f"Top {int(pct*100)}%: n_dark={len(dark)}, n_bright={len(bright)}, p={p:.4f}, r={r:.3f} "
              f"| mean(dark)={dark.mean():.4f}, mean(bright)={bright.mean():.4f}, higher={higher}")

def run_median_split(df, label):
    median_size = df["splice_size_frac"].median()
    df = df.copy()
    df["large_splice_median"] = (df["splice_size_frac"] >= median_size).astype(int)
    model = "iou ~ polarity_bin * large_splice_median + abs_contrast"
    fit = smf.ols(formula=model, data=df).fit(cov_type="HC3")
    print(f"=== {label}, median split (cutoff={median_size:.4f}) ===")
    print(fit.summary().tables[1])
    return df, fit

print("########## ResNet18, TP-only ##########")
resnet_tp, resnet_tp_fit = run_median_split(resnet_tp, "ResNet18 TP-only")
print("\n--- Mann-Whitney ---")
stratified_mannwhitney(resnet_tp, outcome="iou")

print("\n\n########## EfficientNet-B0, TP-only ##########")
effnet_tp, effnet_tp_fit = run_median_split(effnet_tp, "EfficientNet-B0 TP-only")
print("\n--- Mann-Whitney ---")
stratified_mannwhitney(effnet_tp, outcome="iou")

########## ResNet18, TP-only ##########
=== ResNet18 TP-only, median split (cutoff=0.0625) ===
                                       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercept                            0.0408      0.004     10.267      0.000       0.033       0.049
polarity_bin                         0.0004      0.003      0.125      0.900      -0.006       0.006
large_splice_median                  0.1281      0.007     19.458      0.000       0.115       0.141
polarity_bin:large_splice_median    -0.0004      0.010     -0.043      0.965      -0.021       0.020
abs_contrast                       6.91e-05   7.48e-05      0.924      0.356   -7.75e-05       0.000

--- Mann-Whitney ---
Top 40%: n_dark=230, n_bright=401, p=0.5585, r=-0.028 | mean(dark)=0.1962, mean(bright)=0.1849, higher=dark_on_bright
Top 33%: n_dark=180, n_bright=340, p=0.5100, r=-0.035 | me

## Compare full-sample vs. TP-only interaction coefficients
Direct side-by-side check: does the sign and rough magnitude of the
polarity×size interaction survive restricting to true positives?

In [7]:
median_size_full_r = resnet_df["splice_size_frac"].median()
resnet_df["large_splice_median"] = (resnet_df["splice_size_frac"] >= median_size_full_r).astype(int)
fit_full_r = smf.ols("iou ~ polarity_bin * large_splice_median + abs_contrast",
                      data=resnet_df).fit(cov_type="HC3")

median_size_full_e = effnet_df["splice_size_frac"].median()
effnet_df["large_splice_median"] = (effnet_df["splice_size_frac"] >= median_size_full_e).astype(int)
fit_full_e = smf.ols("iou ~ polarity_bin * large_splice_median + abs_contrast",
                      data=effnet_df).fit(cov_type="HC3")

term = "polarity_bin:large_splice_median"
print("Architecture      | Full-sample coef (p)      | TP-only coef (p)")
print(f"ResNet18          | {fit_full_r.params[term]:+.4f} (p={fit_full_r.pvalues[term]:.4f})   "
      f"| {resnet_tp_fit.params[term]:+.4f} (p={resnet_tp_fit.pvalues[term]:.4f})")
print(f"EfficientNet-B0   | {fit_full_e.params[term]:+.4f} (p={fit_full_e.pvalues[term]:.4f})   "
      f"| {effnet_tp_fit.params[term]:+.4f} (p={effnet_tp_fit.pvalues[term]:.4f})")

Architecture      | Full-sample coef (p)      | TP-only coef (p)
ResNet18          | -0.0143 (p=0.1500)   | -0.0004 (p=0.9655)
EfficientNet-B0   | -0.0663 (p=0.0000)   | -0.0483 (p=0.0000)


## Optional — three-way interaction on TP-only pooled data
Restricts the pooled architecture-comparison model to images that are TP
under BOTH classifiers, for the strictest version of the reversal check.

In [8]:
# ============================================================
# STRICT BOTH-TP THREE-WAY INTERACTION
# ============================================================

# Images correctly classified as Tampered by BOTH architectures
both_tp_filenames = (
    set(resnet_df.loc[resnet_df["predicted_label"] == 1, "filename"])
    &
    set(effnet_df.loc[effnet_df["predicted_label"] == 1, "filename"])
)

print(f"Images that are TP under BOTH architectures: {len(both_tp_filenames)}")


# ------------------------------------------------------------
# Keep exactly the same common TP images for both architectures
# ------------------------------------------------------------
res_both = resnet_df[
    resnet_df["filename"].isin(both_tp_filenames)
].copy()

eff_both = effnet_df[
    effnet_df["filename"].isin(both_tp_filenames)
].copy()


# ------------------------------------------------------------
# IMPORTANT:
# Recompute ONE common median size cutoff on the shared subset
# ------------------------------------------------------------
common_median = res_both["splice_size_frac"].median()

res_both["large_splice_median"] = (
    res_both["splice_size_frac"] >= common_median
).astype(int)

eff_both["large_splice_median"] = (
    eff_both["splice_size_frac"] >= common_median
).astype(int)

print(f"Common both-TP median size cutoff: {common_median:.6f}")


# ------------------------------------------------------------
# Sanity checks:
# same images must have identical image-level properties
# ------------------------------------------------------------
check = res_both[
    [
        "filename",
        "polarity_bin",
        "large_splice_median",
        "abs_contrast"
    ]
].merge(
    eff_both[
        [
            "filename",
            "polarity_bin",
            "large_splice_median",
            "abs_contrast"
        ]
    ],
    on="filename",
    suffixes=("_resnet", "_effnet")
)

assert len(check) == len(both_tp_filenames)

assert (
    check["polarity_bin_resnet"].values
    == check["polarity_bin_effnet"].values
).all(), "polarity_bin differs across architectures!"

assert (
    check["large_splice_median_resnet"].values
    == check["large_splice_median_effnet"].values
).all(), "large_splice_median differs across architectures!"

assert np.allclose(
    check["abs_contrast_resnet"].values,
    check["abs_contrast_effnet"].values,
    equal_nan=True
), "abs_contrast differs across architectures!"

print("✓ Shared-image predictors match across architectures.")


# ------------------------------------------------------------
# Architecture labels + image IDs
# ------------------------------------------------------------
res_both["architecture"] = "resnet18"
eff_both["architecture"] = "efficientnet"

res_both["image_id"] = res_both["filename"].astype(str)
eff_both["image_id"] = eff_both["filename"].astype(str)


cols = [
    "image_id",
    "iou",
    "polarity_bin",
    "large_splice_median",
    "abs_contrast",
    "architecture"
]

combined_both_tp = pd.concat(
    [res_both[cols], eff_both[cols]],
    ignore_index=True
)

combined_both_tp["arch_bin"] = (
    combined_both_tp["architecture"] == "efficientnet"
).astype(int)


# ------------------------------------------------------------
# Three-way model with image-clustered standard errors
# ------------------------------------------------------------
if len(both_tp_filenames) >= 50:

    fit_both_tp = smf.ols(
        """
        iou ~ polarity_bin
            * large_splice_median
            * arch_bin
            + abs_contrast
        """,
        data=combined_both_tp
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": combined_both_tp["image_id"]}
    )

    term3 = "polarity_bin:large_splice_median:arch_bin"

    print("\n=== BOTH-TP THREE-WAY INTERACTION ===")
    print(f"Unique images: {len(both_tp_filenames)}")
    print(f"Total observations: {len(combined_both_tp)}")

    print(f"\nCoefficient: {fit_both_tp.params[term3]:.6f}")
    print(f"SE:          {fit_both_tp.bse[term3]:.6f}")
    print(f"p-value:     {fit_both_tp.pvalues[term3]:.6g}")
    print(
        "95% CI:     ",
        tuple(fit_both_tp.conf_int().loc[term3].round(6))
    )

    # Architecture-specific size × polarity effects
    beta_resnet = fit_both_tp.params[
        "polarity_bin:large_splice_median"
    ]

    beta_effnet = (
        fit_both_tp.params[
            "polarity_bin:large_splice_median"
        ]
        +
        fit_both_tp.params[
            "polarity_bin:large_splice_median:arch_bin"
        ]
    )

    print("\n=== ARCHITECTURE-SPECIFIC SIZE × POLARITY EFFECTS ===")
    print(f"ResNet18:      {beta_resnet:+.6f}")
    print(f"EfficientNet:  {beta_effnet:+.6f}")

    print("\n=== FULL COEFFICIENT TABLE ===")
    print(fit_both_tp.summary().tables[1])

else:
    print("Too few shared TP images for reliable pooled analysis.")

Images that are TP under BOTH architectures: 1402
Common both-TP median size cutoff: 0.060984
✓ Shared-image predictors match across architectures.

=== BOTH-TP THREE-WAY INTERACTION ===
Unique images: 1402
Total observations: 2804

Coefficient: -0.057443
SE:          0.013127
p-value:     1.20891e-05
95% CI:      (-0.083171, -0.031714)

=== ARCHITECTURE-SPECIFIC SIZE × POLARITY EFFECTS ===
ResNet18:      +0.001306
EfficientNet:  -0.056137

=== FULL COEFFICIENT TABLE ===
                                                coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------------
Intercept                                     0.0393      0.003     11.292      0.000       0.032       0.046
polarity_bin                                 -0.0003      0.003     -0.109      0.913      -0.006       0.006
large_splice_median                           0.1233      0.007     18.490      0.00